### 1. <u>_PYDANTIC AI_</U>
* _Pydantic AI is a `Python framework for building GenAI agents`._
 * _It provides `strong typing`, `validation`, `structured outputs`, `tools`, `dependency injection`, `streaming`, and `model portability`._
___
#### 1.1 <u>_Advantage of Pydantic AI_</u>
* _A big advantage is that the framework is `vendor-agnostic at the agent level`._
* The same agent can run against different _providers by changing the model object or model string_, _rather than rewriting your whole orchestration layer_
___
#### 1.2 <u>_What is an AI Agent_</u>
```text
        User Request                       A AGENT GENERALLY CONSIST OF:
            |                                    1. Instructions: what role it plays
            v                                    2. Model: which LLM it uses
        +------------------+                     3. Tools: functions/APIs/databases it can call
        |  Agent Brain     |                     4. Memory/State: history, saved facts, external store
        |  (LLM + rules)   |                     5. Output schema: structured response contract
        +------------------+                     6. Planning/Control: how it chooses steps
            | decides                            7. Observability: logs/traces/usage
            +--------------------+
            |                    |
            v                    v
        Use Tool?            Answer Directly
            |                    |
            v                    v
        Call function/API     Return response
            |
            v
        Use result + continue reasoning
```

### 2. <U>_Agent in Pydantic AI_</U>
* _Agent is the main component of the Pydantic AI._
```text
    +------------------------------------------------------+
    |                     Agent                            |
    |------------------------------------------------------|
    | Model / Provider                                     |
    | Instructions                                         |
    | Output Type (Pydantic schema)                        |
    | Tools / Toolsets                                     |
    | Dependencies                                         |
    | Model Settings                                       |
    | Capabilities                                         |
    +------------------------------------------------------+
```

```python

```

#### 2.1 Model & Provider
* _Model = wrapper for a family/API style_
* _Provider = authentication and endpoint handling (API_KEY)_

In [ ]:
from pydantic_ai import Agent
from  pydantic_ai.models.mistral import MistralModel
from  pydantic_ai.providers.mistral import MistralProvider

mistral_agent = Agent(
    model = MistralModel(
        model_name= "A_VALID_MISTRAL_MODEL_NAME",                       # Model
        provider= MistralProvider(api_key= "VALID MISTRAL API KEY")     # Provider
    )
)

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.google import GoogleModel
from pydantic_ai.providers.google import GoogleProvider

from config.settings import settings
gemini_agent = Agent(
    model= GoogleModel(
        model_name= settings.GEMINI_MODEL,
        provider= GoogleProvider(api_key=settings.GEMINI_API_KEY)
    ),
    instructions= "Be consise with the answers.",
)

response = await gemini_agent.run("Who is modi ?")
print(response)


#### 2.2 <u>_system_prompt vs instructions_</u>
```text
        1. instructions
           => What this agent should do right now
           => Current agent behavior / style / task guidance
           => Use cases: task style, explanation depth, tone, formatting, audience level
        2. system_prompt:
           =>  System-level steering that can stay in message history
           =>  Hard guardrails / persistent system rules
           =>  Use Cases : security, compliance, role boundaries, non-negotiable rules

```

##### 2.2.1 <u>_Use with system_prompt_</u>
* _Here you are setting a more **“system-like” rule**_.
* _If this **prompt is in the message history** and you continue the conversation with that history_.
* _it **can continue to influence future runs in a way that instructions are not intended to**_.

In [ ]:
from pydantic_ai import Agent
from  pydantic_ai.models.mistral import MistralModel
from  pydantic_ai.providers.mistral import MistralProvider
from config.settings import settings
mistral_agent1 = Agent(
    model = MistralModel(
        model_name= settings.MISTRAL_MODEL,                                 # Model
        provider= MistralProvider(api_key= settings.MISTRAL_API_KEY)        # Provider

    ),
    system_prompt= """
    You are a bot for the company BridgeLabz.
    Help line number : 111-1111-1111
    """
)
response = await mistral_agent1.run("What is the help line number ?")
print(response)

##### 2.2.2 <u>_Use with Instruction_</u>
* _This agent behaves like a Python tutor for this run._
* _If **later you reuse message history with another agent**, the **earlier instructions are not reused from past messages**_
 * _The new agent’s own instructions are what matter_

In [ ]:
from pydantic_ai import Agent
from  pydantic_ai.models.mistral import MistralModel
from  pydantic_ai.providers.mistral import MistralProvider
from config.settings import settings
mistral_agent2 = Agent(
    model = MistralModel(
        model_name= settings.MISTRAL_MODEL,                                 # Model
        provider= MistralProvider(api_key= settings.MISTRAL_API_KEY)        # Provider

    ),
    instructions= """
    You are a Python tutor.
    Explain concepts step by step.
    Use simple examples first.
    """
)
response2 = await mistral_agent2.run("What is Inheritance ?")
print(response2)

##### 2.2.3 <u>_System prompt vs Instructions_</u>
* _mistral_agent1 : uses system_prompt_
* _mistral_agent2 : uses instructions_
* <u>_If mistral_agent1 has already runed before_</u>:
    * => _Any agent running after that will have the system_prompt context info of the Previous Agent._
* _`note`:=> If strict instruction is defined for that it will not give the info regarding previous agent system_prompt context(e.g, agent2)_
* <u>_e.g,_</u>
  * => _mistral_agent3 will have the info regarding bridgelabz help-line num. even though it's context is not provided with in the agent2_


In [ ]:
mistral_agent3 = Agent(
    model = MistralModel(
        model_name= settings.MISTRAL_MODEL,                                 # Model
        provider= MistralProvider(api_key= settings.MISTRAL_API_KEY)        # Provider
    ),
    system_prompt="you are a simple chat bot"
)
response3 = await mistral_agent3.run("What is the help line number for bridgelabz ?")
print(response3)

___

#### 2.3 <u>_Dynamic Instructions_</u>
* _Pydantic AI also supports dynamic instructions using `@agent.instructions`_
* _This Instructions arealways reevaluated at runtime._

In [ ]:
from dataclasses import dataclass
from pydantic_ai import Agent, RunContext

@dataclass
class UserDeps:
    user_name: str
    skill_level: str

agent = Agent[UserDeps, str](
    "google-gla:gemini-2.5-flash"
)

@agent.instructions
def build_instructions(ctx: RunContext[UserDeps]) -> str:
    return (
        f"You are teaching {ctx.deps.user_name}. "
        f"The learner is at {ctx.deps.skill_level} level. "
        f"Adjust the explanation accordingly."
    )

result = agent.run_sync(
    "Explain decorators in Python.",
    deps=UserDeps(user_name="Debdeepta", skill_level="beginner")
)

print(result.output)

#### 2.4 <u>_Structured Output Validation_</u>
* _This is one of Pydantic AI’s biggest strengths._
* _Pydantic AI `builds a JSON schema` from your type and `validates the model’s response against it`._
* _If the output is invalid, the framework can retry with validation feedback._
```
    User prompt
       |
       v
    Agent(output_type=YourSchema)
       |
       v
    LLM produces structured response
       |
       v
    Pydantic validates against schema
       |
       +--> valid   -> result.output returned to your code
       |
       +--> invalid -> model is told what was wrong and retries
```

##### <u>_Basic e.g,_</u>

In [ ]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from config.settings import settings
from  pydantic_ai.models.mistral import MistralModel
from  pydantic_ai.providers.mistral import MistralProvider

"""
    note -> 1. Fields Names should be meaningful to required field value
            2. A proper field name helps the LLM understand what each field is supposed to contain
            3. We can also Provide Description for the Field if Filed names are not relevant with
               the required field value
"""
# APPROACH 1
class Location(BaseModel):
    country : str
    state: str
    city : str
    pin_code : int
    short_description: str

# APPROACH 2
class LocationDesc(BaseModel):
    a : str = Field(description="Country name")
    b : str = Field(description="State name")
    c : str = Field(description="City name")
    d : str = Field(description= "Pin code")
    e : str = Field(description="Short Description")

agent = Agent(
    model= MistralModel(
        model_name= settings.MISTRAL_MODEL,
        provider= MistralProvider(
            api_key=settings.MISTRAL_API_KEY
        )
    ),
    output_type=LocationDesc,
    instructions="""
    You are an agent to give location details in the given format.
    You Will be asked to give location of events.
    """
)

response = await agent.run("""Where is Belur Math ?""")
print(response)                 # AgentRunResult<> Object
print(type(response))           # O/P -> <class 'pydantic_ai.run.AgentRunResult'>

print(response.output)          # Location<> Object (Actual Valid Schema Object)
print(type(response.output))    # <class '__main__.Location'>

loc = response.output
print(loc.d)             # Accessing Specific Filed


#### 2.5 <u>_Tools/Toolset  Calling & Usage_</u>
* _In Pydantic AI, tool calling means the model can decide to call a Python function instead of answering from text alone._
* _Pydantic AI turns your function signature into a tool schema, sends that schema to the model, validates the model’s arguments, executes the function, and then feeds the result back into the run._

##### 2.5.1 <u>_What is a Tool_ ?</u>
* _A tool is just a normal Python function that the LLM is allowed to call._
    ```text
        User -> LLM
                |
                +--> "I need a tool"
                        |
                        v
                  Python function executes
                        |
                        v
                  result goes back to LLM
                        |
                        v
                  final answer
    ```
    ```text
        tool calling = the runtime action
        tool usage    = how you design and apply tools in your agent
    ```

##### 2.5.2 <u>_REGISTERING TOOLS TO A AGENT_</u>
1. **_`@agent_name.tool_plain`_** :
    * _Used for tools that **do not need context**_

##### _e.g, code_ :


In [ ]:
from pydantic_ai import Agent
from  pydantic_ai.models.mistral import MistralModel
from  pydantic_ai.providers.mistral import MistralProvider
from config.settings import settings
import random

mistral_agent = Agent(
    model = MistralModel(
        model_name= settings.MISTRAL_MODEL,                                 # Model
        provider= MistralProvider(api_key= settings.MISTRAL_API_KEY)        # Provider
    ),
)

@mistral_agent.tool_plain
def roll_dice() -> str:
    """Roll a six-sided die and return the result."""
    return str(random.randint(1, 6))

2.  **_`@agent_name.tool`_** :
    * _Use this when the function **needs access to runtime information through RunContext**._
##### _e.g, code_ :

In [ ]:
from pydantic_ai import Agent, RunContext

gemini_agent = Agent(
    "google-gla:gemini-2.5-flash",
    deps_type=str,
    instructions="You are a dice game assistant."
)

@gemini_agent.tool
def get_player_name(ctx: RunContext[str]) -> str:
    """Get the player's name."""
    return ctx.deps

* _Here the first argument is ctx: RunContext[str]_.
* _That means the tool can access_:
    1. _ctx.deps → the dependencies passed at runtime_
    2. _ctx.agent → the current running agent_

* <u>**_Why this matters_**</u> :
* _Because many real tools need app state, such as_:
    * API keys
    * database clients
    * current user info
    * tenant config
    * feature flags
    * authenticated HTTP clients
    * .... e.t.c



3. **_`Using Agent Constructor`_** :
    * _Instead of decorating a function with @agent.tool or @agent.tool_plain, you can **register tools directly** in the **agent constructor using the tools=[...] argument**._
    * _Pydantic AI supports passing plain functions there, and for more control you can **wrap them with Tool(...)** and **specify** things like whether the function takes context via **takes_ctx=True/False**._
##### _e.g, code_ :

In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent, RunContext, Tool

class UserDeps(BaseModel):
    user_name: str

def calculate_discount(price: float, percent: float) -> float:
    """Calculate discounted price."""
    return round(price - (price * percent / 100), 2)

def get_user_name(ctx: RunContext[UserDeps]) -> str:
    """Return the current user's name."""
    return ctx.deps.user_name

agent = Agent(
    "google-gla:gemini-2.5-flash",
    instructions="""
    You are a shopping assistant.
    Use tools when needed.
    """,
    tools=[
        calculate_discount,                  # plain function
        Tool(get_user_name, takes_ctx=True)  # explicit Tool wrapper
    ],
)

#### 2.6 <u>_AGENT WITH DEPENDENCIES_</u>
* _An **agent with dependency** means you define a **dependency type for the whole agent run**, and then pass an instance of that dependency object when you call run() or run_sync()._
* _The dependency object is then **available anywhere that receives a RunContext**, **such as context-aware tools** and **other runtime-aware parts of the agent**._
* _Pydantic AI agent dependencies can be used to provide **runtime services** like **database access**, **HTTP clients**, and stored results that tools can share_
```text
    User request
        |
        v
    Agent
        |
        +--> deps object injected for this run
                |
                +--> tool 1 can use it
                +--> tool 2 can use it
                +--> other runtime-aware logic can use it
```

In [1]:
from pydantic import BaseModel
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.mistral import MistralModel
from pydantic_ai.providers.mistral import MistralProvider
from config.settings import settings

# Step 1: define the dependency type
class MyNotes(BaseModel):
    student_name: str
    notes : dict[str,str]

# Step 2: create the agent and tell it what dependency type it expects
"""
    1. During Agent Creation
        general_syntax -> Agent[Dependecy_type, output_type]
"""
mistral_agent = Agent[MyNotes,str]( # <--------- Dependency type = StudentDeps
    MistralModel(                                # final output type = str
        model_name= settings.MISTRAL_MODEL,
        provider= MistralProvider(
            api_key=settings.MISTRAL_API_KEY
        )
    ),
     instructions="""
    You must answer only from the retrieved note.
    Do not use outside knowledge.
    If the note is missing, say "No note found."
    If possible, return the note text exactly
    """
)

@mistral_agent.tool
def get_note(ctx:RunContext[MyNotes], topic:str) -> str:
    """Return the exact stored note for the topic."""
    return ctx.deps.notes.get(topic.lower(), "No note found.")


# CREATING SAMPLE DATA FOR DEPENDENCY
deps = MyNotes(
    student_name="Debdeepta",
    notes={
        "rag": "RAG means retrieve relevant data first, then generate the answer.",
        "embedding": "Embeddings convert text into vectors that capture meaning."
    }
)
"""
2. While running the agent :
    agent_name.run(
        deps = your_dependecy_object_name
    )
"""
result = await mistral_agent.run(
    "Explain rag using my stored note.",
    deps=deps        # <--------------- INJECTING DEPENDENCY
)

print(result.output)



RAG means **retrieve** relevant data first, then **generate** the answer.


##### 2.6.1 <u>_TOOL ARGUMENTS VS AGENT DEPENDENCIES_</u>
```text
    NORMAL TOOL ARGUMENTS
    ---------------------
    - part of tool schema
    - supplied by model
    - visible as callable inputs
    - good for topic, numbers, options, search terms

    AGENT DEPENDENCIES
    ------------------
    - injected once per run via deps=...
    - accessed through ctx.deps
    - supplied by application
    - good for API clients, DB handles, auth, user/session state, shared tool state
```